In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
from edgar import *
import os
from dotenv import load_dotenv
load_dotenv()
set_identity(os.getenv("EMAIL"))
print("All imported")

All imported


In [ ]:
def get_marketcap(c_name):
    ticker = yf.Ticker("AAPL")
    market_cap = ticker.info.get("marketCap")
    shares = ticker.info.get("sharesOutstanding")
    price = ticker.history(period="1d")["Close"].iloc[-1]
    print(f"Market cap is {market_cap:,.0f}, with {shares} shares valued at {price:.1f}$")
    
    return market_cap

def get_cashdata(c_name):
    c = Company(c_name)              # ticker
    fin = c.get_financials()         # pulls latest 10-K/10-Q financials from XBRL
    cf = fin.cash_flow_statement()   # cash flow statement (DataFrame-like)
    df= cf.to_dataframe()
    row= df.iloc[15]
    a= row.iloc[3]
    return a


#get data of company outside to compute faster
CF= get_cashdata(c_name)
def pricing_dfc(CF,r=.1, n=10, g=.05):
    DCF = CF
    for i in range (n):
        DCF += (CF*(1+g)**i)/(1+r)**i
    
    #print(DCF)
    
    #print(f"future value is {DCF/1000000000:.1f} Billions")
    return DCF

In [92]:

c_name= "AAPl"
years= 1
r=.1
val= int(0)
marketcap= get_marketcap(c_name)
cashflow= get_cashdata(c_name)

def check_years(cashflow, marketcap):
    years= 1
    for i in range(15):
        val= pricing_dfc(cashflow, .07, years) 
        if val/marketcap > 1:
            print(f"At curent cashflow the company is valued at {years} years of cash flow")
            return 
    
        if years>9:
            print(f"At curent cashflow the company is valued at {val/marketcap*100:.2f}% of DCF after 10 years ")
            return 
            
        years += 1 
        
        
def check_rate(cashflow, marketcap):   
    g=.1
    r=.05
    for i in range(200):
        val= pricing_dfc(cashflow, .05, 10, g) 
        g += .01 
        if val/marketcap > 1:
            print(f"At curent cashflow the company is valued at {g*100:.0f}% rate of growth after 10 years")
            return  
        #print(r)
        #print(val/marketcap*100)    
        g += .01
        
        if g>1:
            print(f"At curentt cashflow with 100% growth the comapny is at {val/marketcap*100:.0f}% of marketcap ")
            return 
            


    
   
    
    
    
    

Market cap is 3,759,141,289,984, with 14681140000 shares valued at 255.8$


In [84]:
c_name= "AAPl"
apple_marketcap= get_marketcap(c_name)
apple_cashflow= get_cashdata(c_name)

check_years(apple_cashflow, apple_marketcap) 
check_rate(apple_cashflow, apple_marketcap) 


Market cap is 3,759,141,289,984, with 14681140000 shares valued at 255.8$
At curent cashflow the company is valued at 25.25% of DCF after 10 years 
At curentt cashflow with 100% growth the comapny is at 9% of marketcap 


In [93]:
c_name= "AAPl"
apple_marketcap= get_marketcap(c_name)
apple_cashflow= get_cashdata(c_name)

check_years(apple_cashflow, apple_marketcap) 
check_rate(apple_cashflow, apple_marketcap) 

Market cap is 3,759,141,289,984, with 14681140000 shares valued at 255.8$
At curent cashflow the company is valued at 30.25% of DCF after 10 years 
At curent cashflow the company is valued at 27% rate of growth after 10 years
